# Functional characterization (supporting)

Supporting functional context for the differential-expression results, in two parts:

1. **Candidate-set over-representation on the primary dorsolateral result.** A Fisher exact test of
   whether the projection candidates (Set B) and the 118-gene neuromodulation panel
   (Set A) are enriched among the dorsolateral-vs-ventromedial significant genes.
2. **Pathway enrichment on the anteroposterior regional axis.** Enrichr over-representation
   (GO Biological Process, KEGG mouse, Reactome) on the posterior/anterior robustness contrast, kept
   as supporting characterization of the regional signature rather than the headline. One direction
   returns no terms, which is reported as a null, not interpreted.

**Inputs:** `results/tables/de_dorsolateral_vs_ventromedial_deseq2.csv` (primary),
`results/tables/de_posterior_vs_anterior_deseq2.csv` (regional).

In [1]:
from pathlib import Path

import gseapy as gp
import pandas as pd
import plotting_style as ps
import yaml

ROOT = Path("..")
with open(ROOT / "config/config.yaml") as fh:
    cfg = yaml.safe_load(fh)
paths = cfg["paths"]

CANDIDATE_GENES = ROOT / paths["candidate_genes"]

OUTPUT_DIR = ROOT / paths["results_dir"]
OUTPUT_DIR.mkdir(exist_ok=True)
FIG_DIR = OUTPUT_DIR / "figures_raw"
TAB_DIR = OUTPUT_DIR / "tables"
FIG_DIR.mkdir(exist_ok=True)
TAB_DIR.mkdir(exist_ok=True)
FDR, LOG2FC = cfg["de"]["fdr"], cfg["de"]["log2fc"]
GENE_SETS = cfg["enrichment"]["gene_sets"]
ORGANISM = cfg["enrichment"]["organism"]

ps.apply_style()

## Candidate-set over-representation (primary dorsolateral result)

A Fisher exact test asks whether each candidate gene set is enriched among the
dorsolateral-vs-ventromedial significant genes more than expected by chance, run against the tested
background using the primary DE result.

The background is the set of unique gene symbols in the primary DE result (19,017), slightly fewer
than the 19,024 tested gene identifiers (`04_differential_expression.ipynb`) because a few Ensembl
IDs carry no symbol or share one. The candidate sets are matched by symbol.

In [2]:
from scipy.stats import fisher_exact

dl = pd.read_csv(TAB_DIR / "de_dorsolateral_vs_ventromedial_deseq2.csv").dropna(subset=["symbol"])
dl["symbol"] = dl["symbol"].astype(str)
dl_sig = set(dl.loc[(dl["padj"] < FDR) & (dl["log2FoldChange"].abs() > LOG2FC), "symbol"])
background = set(dl["symbol"])
n_bg, n_sig = len(background), len(dl_sig)

table12 = {"Robo2", "Abi3bp", "Gabrg1", "Adcyap1", "Chrm3", "Rprm", "Thrb", "Cntn5"}
setA = set(pd.read_csv(CANDIDATE_GENES)["GeneName"].dropna().astype(str))

def ora(name, genes):
    """One-sided Fisher over-representation of a gene set among the significant DE genes."""
    tested = genes & background
    hits = tested & dl_sig
    a = len(hits)
    b = len(tested) - a
    c = n_sig - a
    d = (n_bg - n_sig) - b
    odds, p = fisher_exact([[a, b], [c, d]], alternative="greater")
    expected = len(tested) * n_sig / n_bg
    print(f"{name}: {a}/{len(tested)} tested candidates significant "
          f"(expected ~{expected:.1f}), odds ratio {odds:.1f}, Fisher p = {p:.1e}")
    return {"set": name, "n_tested": len(tested), "n_sig": a,
            "expected": round(expected, 2), "odds_ratio": round(odds, 2), "fisher_p": p}

# Background is unique gene symbols (a few of the tested gene identifiers share or lack a symbol),
# since the candidate sets are matched by symbol.
print(f"Background: {n_bg} tested gene symbols, {n_sig} significant at FDR<{FDR} & |log2FC|>{LOG2FC}\n")
ora_rows = [ora("Set B (8 genes)", table12),
            ora("Set A (118-gene panel)", setA)]
pd.DataFrame(ora_rows).to_csv(TAB_DIR / "candidate_set_ora.csv", index=False)

Background: 19017 tested gene symbols, 1347 significant at FDR<0.05 & |log2FC|>1.0

Set B (8 genes): 5/8 tested candidates significant (expected ~0.6), odds ratio 21.9, Fisher p = 8.3e-05
Set A (118-gene panel): 27/108 tested candidates significant (expected ~7.6), odds ratio 4.4, Fisher p = 5.5e-09


## Load DE result and define up/down gene sets

Significant genes (FDR < 0.05, |log2FC| > 1) are split by direction. Enrichment is run on
each direction separately.

In [3]:
de = pd.read_csv(TAB_DIR / "de_posterior_vs_anterior_deseq2.csv")
sig = de[(de["padj"] < FDR) & (de["log2FoldChange"].abs() > LOG2FC)].dropna(subset=["symbol"])

up = sorted(sig.loc[sig["log2FoldChange"] > 0, "symbol"].astype(str).unique())
down = sorted(sig.loc[sig["log2FoldChange"] < 0, "symbol"].astype(str).unique())
print(f"Up in posterior: {len(up)} genes | Up in anterior: {len(down)} genes")

Up in posterior: 674 genes | Up in anterior: 1172 genes


## Enrichr over-representation

Each direction is queried against all three libraries and filtered to adjusted p < 0.05.
Saved tables are reused if present, so re-running is reproducible offline. A fresh query
requires network access (Enrichr is a web service).

In [4]:
def run_enrichr(genes, tag):
    """Enrichr over-representation for a gene-symbol list across GENE_SETS.

    Reuses the saved table if present so re-runs are reproducible offline. Otherwise
    queries Enrichr and writes the full table to CSV. Returns the significant results
    (adjusted p < 0.05).
    """
    csv = TAB_DIR / f"enrichment_{tag}.csv"
    if csv.exists():
        res = pd.read_csv(csv)
    elif len(genes) < 5:
        print(f"{tag}: too few genes to test")
        return pd.DataFrame()
    else:
        try:
            enr = gp.enrichr(gene_list=list(genes), gene_sets=GENE_SETS,
                             organism=ORGANISM, outdir=None)
        except Exception as exc:
            print(f"{tag}: Enrichr query failed ({exc})")
            return pd.DataFrame()
        res = enr.results.copy()
        res["direction"] = tag
        res.to_csv(csv, index=False)
    hits = res[res["Adjusted P-value"] < 0.05].sort_values("Adjusted P-value")
    print(f"{tag}: {len(hits)} enriched terms (adj p < 0.05) across {res['Gene_set'].nunique()} libraries")
    return hits


up_hits = run_enrichr(up, "posterior_up")
down_hits = run_enrichr(down, "anterior_up")
for tag, hits in [("posterior_up", up_hits), ("anterior_up", down_hits)]:
    if len(hits):
        print(f"\nTop terms {tag}:")
        print(hits.head(8)[["Gene_set", "Term", "Adjusted P-value", "Overlap"]].to_string(index=False))

posterior_up: 0 enriched terms (adj p < 0.05) across 3 libraries
anterior_up: 15 enriched terms (adj p < 0.05) across 3 libraries

Top terms anterior_up:
       Gene_set                                                            Term  Adjusted P-value Overlap
KEGG_2019_Mouse                         Neuroactive ligand-receptor interaction          0.000053  46/348
KEGG_2019_Mouse                                              Pathways in cancer          0.000135  60/535
  Reactome_2022                      Defective B3GALTL Causes PpS R-HSA-5083635          0.003655   11/37
  Reactome_2022 O-glycosylation Of TSR Domain-Containing Proteins R-HSA-5173214          0.003655   11/38
KEGG_2019_Mouse                                                   Breast cancer          0.004013  22/147
  Reactome_2022                     Amine Ligand-Binding Receptors R-HSA-375280          0.004201   11/40
KEGG_2019_Mouse                                          cAMP signaling pathway          0.017764  26/21

## Top enriched terms (anteroposterior regional axis, supporting)

Anterior-up genes enrich for neuroactive ligand-receptor signalling, amine receptors, and
cAMP/calcium signalling. Posterior-up genes return no terms at FDR < 0.05. A null enrichment result
is reported as such and not interpreted as a transcription-factor or identity signature.

In [5]:
fig = ps.enrichment_dotplot(down_hits, up_hits, len(down), len(up), fdr=FDR)
ps.save_fig(fig, FIG_DIR / "enrichment_dotplot")

print(f"Anterior: {len(down_hits)} significant terms | "
      f"Posterior: {len(up_hits)} significant terms (FDR < {FDR}). Supporting characterization of "
      f"the anteroposterior regional axis; a null direction is not over-interpreted.")


Anterior: 15 significant terms | Posterior: 0 significant terms (FDR < 0.05). Supporting characterization of the anteroposterior regional axis; a null direction is not over-interpreted.
